# 📋 claude.ai 프롬프트 / claude.ai Prompt

> **수업 활용법:** 아래 프롬프트를 [claude.ai](https://claude.ai)에 붙여넣으면 유사한 노트북을 생성할 수 있습니다.

---

### 🇰🇷 한국어 프롬프트
```
Anthropic Claude API + Wikipedia 무료 API를 이용한
다단계 리서치 AI 에이전트 Google Colab 노트북을 한국어로 작성해주세요.

1. 개념: '상태 기억 에이전트' — 도구 호출 간 상태(노트)를 유지하며
   여러 단계에 걸쳐 정보를 수집·통합하는 패턴 설명
2. 도구 3개:
   - search_wikipedia(query): 한국어/영어 Wikipedia REST API 검색
   - save_note(topic, content): 연구 노트를 딕셔너리에 저장 (상태 유지)
   - get_all_notes(): 저장된 모든 노트 반환
3. 에이전트 루프: Claude가 여러 번 Wikipedia 검색 후 노트 저장,
   마지막에 get_all_notes()로 통합하여 최종 보고서 작성
4. 테스트 1: 특정 기업(예: '구글')을 조사하여 역사·사업·경쟁사 비교 보고서
5. 테스트 2: AI 관련 주제(예: 'ChatGPT vs Claude')를 비교 분석
6. 미니 실습: input()으로 주제 입력받아 조사 보고서 생성

설치: anthropic, requests. 한국어 주석. 초보자 친화적.
```

### 🇺🇸 English Prompt
```
Create a Korean Google Colab notebook for a Multi-step Research AI Agent
using Anthropic Claude API + Wikipedia free REST API.

1. Concept: 'Stateful Agent' — maintains notes across tool calls,
   collects information in multiple steps, then synthesizes
2. Three tools:
   - search_wikipedia(query): Korean/English Wikipedia REST API
   - save_note(topic, content): save to in-memory dict (stateful)
   - get_all_notes(): retrieve all saved notes
3. Agent loop: Claude searches Wikipedia multiple times, saves notes,
   then calls get_all_notes() to synthesize a final report
4. Test 1: Research a company and write a structured report
5. Test 2: Compare two AI topics
6. Mini exercise: input() for custom topic

Install: anthropic, requests. Korean comments. Beginner-friendly.
```

# FM2 실습 5: 다단계 리서치 AI 에이전트

## 학습 목표
- **상태 기억 에이전트** 패턴을 이해하고 구현한다
- Wikipedia 무료 API를 도구로 연동하여 실제 정보를 수집한다
- 에이전트가 여러 단계에 걸쳐 정보를 수집·통합하는 과정을 경험한다

## 이번 실습의 새로운 개념: 상태 기억 에이전트

| FM2-02 에이전트 | FM2-05 에이전트 |
|----------------|----------------|
| 1~2번 도구 호출 후 바로 답변 | 여러 번 검색 → 노트 저장 → 통합 |
| 상태 없음 (기억 없음) | **상태 유지**: 노트에 누적 저장 |
| 단순 조회 | 다단계 계획·실행 |

## 에이전트 동작 흐름
```
"구글의 역사, 사업, 경쟁사를 조사해줘"
    ↓
[Step 1] search_wikipedia("구글") → 기본 정보 수집
    ↓
[Step 2] save_note("회사 개요", "...")
    ↓
[Step 3] search_wikipedia("구글 검색 엔진") → 사업 세부 정보
    ↓
[Step 4] save_note("주요 사업", "...")
    ↓
[Step 5] search_wikipedia("마이크로소프트") → 경쟁사 조사
    ↓
[Step 6] save_note("경쟁사 비교", "...")
    ↓
[Step 7] get_all_notes() → 전체 노트 조합
    ↓
최종 보고서 생성
```

In [ ]:
!pip install anthropic requests -q
import getpass, anthropic
import requests
import urllib.parse

api_key = getpass.getpass("🔑 Anthropic API 키: ")
client = anthropic.Anthropic(api_key=api_key)
print("✅ 준비 완료!")

## 1단계: 도구 3개 정의

### Wikipedia API 안내
- **무료**, API 키 불필요
- 엔드포인트: `https://ko.wikipedia.org/api/rest_v1/page/summary/{제목}`
- 한국어 Wikipedia에 없으면 영어 Wikipedia로 자동 시도

In [ ]:
# ── 도구 1: Wikipedia 검색 ─────────────────────────────────────
def search_wikipedia(query: str) -> str:
    """
    Wikipedia에서 주제를 검색하고 요약을 반환합니다.
    한국어 Wikipedia를 먼저 시도하고, 없으면 영어 Wikipedia를 사용합니다.
    """
    encoded = urllib.parse.quote(query)

    # 한국어 Wikipedia 시도
    for lang in ['ko', 'en']:
        url = f"https://{lang}.wikipedia.org/api/rest_v1/page/summary/{encoded}"
        try:
            resp = requests.get(url, timeout=10,
                                headers={'User-Agent': 'FM2-Research-Agent/1.0'})
            if resp.status_code == 200:
                data = resp.json()
                title   = data.get('title', query)
                extract = data.get('extract', '내용 없음')
                # 너무 길면 600자로 자름
                if len(extract) > 600:
                    extract = extract[:600] + "..."
                lang_label = "한국어" if lang == 'ko' else "영어"
                return f"[Wikipedia {lang_label}] {title}\n{extract}"
        except Exception:
            continue

    return f"'{query}'에 대한 Wikipedia 정보를 찾을 수 없습니다."


# ── 도구 2: 노트 저장 (상태 유지의 핵심) ─────────────────────────
# 이 딕셔너리가 에이전트의 '기억'입니다
research_notes = {}

def save_note(topic: str, content: str) -> str:
    """
    조사한 내용을 노트에 저장합니다.
    에이전트가 여러 번 검색한 결과를 누적 기록하는 데 사용합니다.
    """
    research_notes[topic] = content
    return f"✅ '{topic}' 노트 저장 완료 (총 {len(research_notes)}개 노트)"


# ── 도구 3: 저장된 노트 전체 조회 ────────────────────────────────
def get_all_notes() -> str:
    """
    저장된 모든 연구 노트를 반환합니다.
    최종 보고서 작성 전에 호출하여 수집된 정보를 통합합니다.
    """
    if not research_notes:
        return "저장된 노트가 없습니다."

    result = [f"📚 저장된 연구 노트 ({len(research_notes)}개)\n" + "="*50]
    for i, (topic, content) in enumerate(research_notes.items(), 1):
        result.append(f"\n[노트 {i}] {topic}\n{content}")
    return "\n".join(result)


# 도구 테스트
print("도구 테스트:")
print(search_wikipedia("인공지능")[:200], "...\n")
print(save_note("테스트", "테스트 내용"))
research_notes.clear()  # 테스트 노트 초기화
print("\n✅ 도구 3개 준비 완료!")

In [ ]:
# ─── 도구 스키마 정의 ─────────────────────────────────────────
TOOLS = [
    {
        "name": "search_wikipedia",
        "description": (
            "Wikipedia에서 주제를 검색하고 요약 정보를 반환합니다. "
            "회사, 인물, 기술, 사건 등 어떤 주제든 조사할 수 있습니다."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "검색할 주제 또는 키워드 (예: '구글', '인공지능', 'ChatGPT')"
                }
            },
            "required": ["query"]
        }
    },
    {
        "name": "save_note",
        "description": (
            "조사한 내용을 주제별로 노트에 저장합니다. "
            "나중에 get_all_notes()로 통합하여 보고서를 작성할 때 사용합니다."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "topic":   {"type": "string", "description": "노트 주제 (예: '회사 개요', '주요 사업')"},
                "content": {"type": "string", "description": "저장할 내용"}
            },
            "required": ["topic", "content"]
        }
    },
    {
        "name": "get_all_notes",
        "description": (
            "지금까지 저장한 모든 연구 노트를 반환합니다. "
            "충분한 정보를 수집했다고 판단될 때 호출하여 최종 보고서를 작성하세요."
        ),
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
]

TOOL_FUNCTIONS = {
    "search_wikipedia": search_wikipedia,
    "save_note":        save_note,
    "get_all_notes":    get_all_notes,
}

print("✅ 도구 스키마 3개 준비 완료!")

## 2단계: 리서치 에이전트 루프

In [ ]:
RESEARCH_SYSTEM_PROMPT = """당신은 체계적으로 정보를 조사하는 리서치 에이전트입니다.

조사 방법:
1. 주제와 관련된 여러 키워드로 Wikipedia를 2~4회 검색합니다
2. 검색할 때마다 핵심 내용을 save_note()로 저장합니다
3. 충분한 정보가 모이면 get_all_notes()로 전체를 불러옵니다
4. 수집한 정보를 통합하여 마크다운 형식의 보고서를 작성합니다

보고서 형식:
# 📊 [주제] 조사 보고서

## 주요 발견 사항
...

## 세부 분석
...

## 결론
..."""


def run_research_agent(topic, verbose=True):
    """리서치 에이전트를 실행합니다."""
    global research_notes
    research_notes = {}  # 매 실행마다 노트 초기화

    messages = [{"role": "user", "content": topic}]

    if verbose:
        print(f"\n📋 조사 주제: {topic}")
        print("─" * 65)

    step = 0
    while True:
        step += 1
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=2048,
            system=RESEARCH_SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages
        )

        messages.append({"role": "assistant", "content": response.content})

        # 최종 보고서
        if response.stop_reason == "end_turn":
            final = next((b.text for b in response.content if hasattr(b, 'text')), "")
            print(final)
            if verbose:
                print(f"\n📊 총 {step}번 API 호출, {len(research_notes)}개 노트 수집")
            return final

        # 도구 사용
        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue

                tool_input = block.input  # ✅ block.input 사용

                if verbose:
                    args_str = ", ".join(f"{k}={repr(v)[:30]}" for k, v in tool_input.items())
                    print(f"  [Step {step}] 🔧 {block.name}({args_str})")

                func   = TOOL_FUNCTIONS[block.name]
                result = func(**tool_input) if tool_input else func()

                if verbose and block.name == "search_wikipedia":
                    preview = result.replace('\n', ' ')[:80]
                    print(f"             📖 {preview}...")
                elif verbose:
                    print(f"             ✅ {result}")

                tool_results.append({
                    "type":        "tool_result",
                    "tool_use_id": block.id,
                    "content":     str(result)
                })

            messages.append({"role": "user", "content": tool_results})


print("✅ 리서치 에이전트 준비 완료!")

## 3단계: 테스트

In [ ]:
# 테스트 1: 기업 조사 보고서
# 에이전트가 Wikipedia를 여러 번 검색하고 노트를 쌓아가는 과정을 관찰하세요
run_research_agent(
    "구글(Google)의 역사, 주요 사업 분야, 주요 경쟁사에 대해 조사하고 보고서를 작성해줘"
)

In [ ]:
# 테스트 2: AI 모델 비교 분석
run_research_agent(
    "ChatGPT와 Claude AI를 비교 분석해줘. 각각의 특징, 장단점, 개발사 정보를 포함해줘."
)

## ✏️ 미니 실습: 직접 조사 주제 입력하기

In [ ]:
print("📋 리서치 에이전트 — 주제를 입력하면 Wikipedia를 조사하여 보고서를 작성합니다.")
print("예시: '테슬라', 'K-팝의 세계화', '양자컴퓨터 vs 고전컴퓨터'\n")

my_topic = input("조사할 주제를 입력하세요: ").strip()

if my_topic:
    run_research_agent(f"{my_topic}에 대해 조사하고 핵심 내용을 정리한 보고서를 작성해줘")
else:
    print("주제를 입력해주세요.")

## 📝 FM2 에이전트 시리즈 전체 정리

### 5개 예제의 단계별 학습 로드맵

| 실습 | 핵심 개념 | 도구 수 | 특징 |
|------|----------|---------|------|
| **FM2-02** Tool Use 기초 | 에이전트 루프, stop_reason | 3개 | 기본 패턴 입문 |
| **FM2-03** 업무 자동화 | 멀티 API 파이프라인 | 0개 (순차 호출) | 비즈니스 자동화 |
| **FM2-04** 뉴스 에이전트 | 외부 데이터 연동 | 1개 (RSS 검색) | 실시간 정보 수집 |
| **FM2-05** 리서치 에이전트 | 상태 기억, 다단계 계획 | 3개 (검색·저장·조합) | 정보 수집·통합 |

### 에이전트 설계 핵심 원칙

1. **도구는 단순하게**: 각 도구는 하나의 명확한 역할만
2. **상태는 명시적으로**: 노트처럼 저장·조회 도구를 제공
3. **시스템 프롬프트로 행동 유도**: 에이전트의 작업 방식을 명확히 지시
4. **stop_reason 확인**: `tool_use` ↔ `end_turn` 두 가지만 처리
5. **block.input 사용**: `block.tool_input` ❌ → `block.input` ✅

### 실제 적용 아이디어
- 경쟁사 제품 조사 자동화
- 신기술 동향 모니터링 + 요약 보고서
- 신입사원 온보딩 자료 자동 생성
- 회의 전 참석자·안건 사전 조사